# 5. Fluorescence Anisotropy Simulation

This notebook extends the PIE photon-timestamp simulation of `2_PIE_Timestamps_Simulation.ipynb` with **polarization-resolved photon streams**, using the time-resolved lifetime simulation of `3_Dynamic_State_Exchange_Simulation.ipynb` as a template.

The physical model is deliberately the simplest possible:

1. **Pulsed excitation** (20 MHz, as in the PIE simulations) with the excitation polarization along the lab $z$ axis. A dye absorbs a photon with probability $\propto \cos^2\theta$, so the excited-state transition dipole orientation $\hat{u}_0$ is distributed as $\propto \cos^2\theta$.
2. **Wobbling dye**: the dye is flexibly linked to the molecule and reorients by isotropic Brownian rotational diffusion. While the fluorophore is in the excited state it rotates for the fluorescence delay time $t$ (drawn from an exponential decay, convolved with the IRF, as in notebook 3). The rotational diffusion constant is $D_{rot} = 1/(6\,\tau_{rot})$, where $\tau_{rot}$ is the rotational correlation time, so the ensemble anisotropy follows the Perrin equation $r(t) = r_0\,e^{-t/\tau_{rot}}$ with $r_0 = 0.4$.
3. **Polarized detection**: the emission dipole radiates into parallel and perpendicular polarization analyzers. Photons are detected along the lab $x$ axis with the analyzers along $z$ (parallel) and $y$ (perpendicular). For an emission dipole $\hat{u}$ the relative intensities are $I_{\parallel} \propto u_z^2$ and $I_{\perp} \propto u_y^2$, so each photon is sorted into the parallel or perpendicular stream with probability

$$P(\parallel\,|\,\hat{u}) = \frac{u_z^2}{u_z^2 + u_y^2}.$$

At $t = 0$ this gives $r_0 = (I_\parallel - I_\perp)/(I_\parallel + 2I_\perp) = 0.4$, and after rotational diffusion the time-resolved anisotropy decays as $r(t) = 0.4\,e^{-t/\tau_{rot}}$.

The result is a photon-HDF5 file with **four photon streams**: donor-parallel, donor-perpendicular, acceptor-parallel and acceptor-perpendicular (the acceptor streams separated into DexAem and AexAem via the nanotime windows, as in notebook 3).


In [ ]:
%matplotlib inline
import numpy as np
import tables
import matplotlib.pyplot as plt
import seaborn as sns
import pybromo as pbm
import phconvert as phc
import pandas as pd
import fretbursts as fb
print('Numpy version:', np.__version__)
print('PyTables version:', tables.__version__)
print('PyBroMo version:', pbm.__version__)
print('phconvert version:', phc.__version__)


def patch_asscalar(a):
    return a.item()

setattr(np, "asscalar", patch_asscalar)


## 1. Load the PyBroMo diffusion simulation

As in `2_PIE_Timestamps_Simulation.ipynb`, the diffusion trajectory from `1_Trajectory_Simulation.ipynb` sets *where* and *when* molecules traverse the confocal volume. The single-particle emission rate through the point-spread function is simulated with `pbm.TimestampSimulation`.


In [ ]:
sim_id = "aa3ff9" # The simulation identifier, first six characters of the PyBroMo trajectory
S = pbm.ParticlesSimulation.from_datafile(sim_id)
rs = np.random.RandomState()

# Trajectory Parameters
number_of_particles = S.num_particles # Total number of simulated particles
t_step = S.t_step # Simulation time step in seconds


In [ ]:
# User settings
fret_countrate = 350E3       # Total count rate of DA
acceptor_countrate = 350E3   # Total count rate of the acceptor
donor_channel_bg = 1500       # Donor channel background count rate in Hz
acceptor_channel_bg = 1500    # Acceptor channel background count rate in Hz


In [ ]:
# Simulate the photon emission timestamps of a single FRET species through the PSF,
# exactly as in 2_PIE_Timestamps_Simulation.ipynb
E_fret = 0.5

fret_sim = pbm.TimestampSimulation(
    S,
    num_particles = (number_of_particles, ),
    em_rates = (fret_countrate, ),
    E_values = (E_fret, ),
    bg_rate_d = donor_channel_bg,
    bg_rate_a = acceptor_channel_bg
)
fret_sim.run(rs, overwrite=True, skip_existing=True, save_pos=False)

# The PIE partner: acceptor-only excitation (AexAem), as in notebook 2
a_only_sim = pbm.TimestampSimulation(
    S,
    num_particles = (number_of_particles, ),
    em_rates = (acceptor_countrate, ),
    E_values = (1.0, ),
    bg_rate_d = donor_channel_bg,
    bg_rate_a = acceptor_channel_bg
)
a_only_sim.run(rs, overwrite=True, skip_existing=True, save_pos=False)

print(fret_sim.m_filenames)
print(a_only_sim.m_filenames)


## 2. Time-resolved excitation: IRF and lifetime draws

Directly taken from `3_Dynamic_State_Exchange_Simulation.ipynb`: the fluorescence decay is simulated by drawing exponentially distributed lifetimes and convolving them with the measured instrument response function. If the IRF csv file of notebook 3 is not available, a synthetic Gaussian IRF is used instead. The anisotropy decay happens *during* this delay time, which links the rotational diffusion to the nanotimes.


In [ ]:
irf_path = '../../simulations/harris_divisor_example/532nm_IRF_19-10-2021.csv'
try:
    irf_read = pd.read_csv(irf_path, header = 5).to_numpy().reshape(-1)
    irf_read -= 20
    irf_read[irf_read < 0] = 0
except FileNotFoundError:
    # Synthetic Gaussian IRF centered on bin 270 (matching the nanotime of notebook 3)
    irf_read = np.exp(-0.5*((np.arange(4096) - 270)/15.0)**2)
irf_cum = irf_read.cumsum()
IRF_CDF = irf_cum/irf_cum[-1]
IRF_CENTER = int(np.argmax(irf_read)) # IRF peak bin

# Nanotime clock: 4096 bins over the 20 MHz laser period
NUM_BINS = 4096
LASER_RATE = 20E6
TCSPC_UNIT = (1/LASER_RATE)/NUM_BINS      # seconds per bin
TCSPC_UNIT_NS = TCSPC_UNIT*1E9            # ns per bin

def rand_nanotimes(tau_ns, size, rs):
    """
    Draw fluorescence delay times (ns) from an exponential decay of lifetime tau_ns,
    convolve with the IRF and convert to nanotime clock bins.
    Returns (nanotimes, delay_ns): the TCSPC bin and the physical delay time,
    which is also the time the dye has to wobble before emitting.
    """
    delay = rs.exponential(scale = tau_ns, size = size) # ns
    irf = np.searchsorted(IRF_CDF, rs.rand(size)).astype(np.int64)
    nano = (irf + np.rint(delay/TCSPC_UNIT_NS).astype(np.int64)) % NUM_BINS
    return nano.astype(np.uint16), delay


## 3. The physical anisotropy model

Three building blocks:

1. `sample_excited_orientations`: absorption of a $z$-polarized photon orients the excited transition dipole $\propto \cos^2\theta$ (rejection-free inverse-CDF sampling, $\cos\theta = (2x-1)^{1/3}$).
2. `rot_diffuse`: the exact single-time propagator of isotropic rotational diffusion over the delay time $t$: with probability $e^{-6 D_{rot} t} = e^{-t/\tau_{rot}}$ the emission dipole is still correlated with the absorption orientation, otherwise it has thermalized to an isotropic orientation. This makes the ensemble anisotropy decay exactly as the Perrin equation $r(t) = 0.4\, e^{-t/\tau_{rot}}$ with $\tau_{rot} = 1/(6 D_{rot})$.
3. `sort_polarization`: a dipole radiates no light along its own axis. The detector views along the lab $x$ axis with the analyzers along $z$ (parallel) and $y$ (perpendicular), so a collected photon has $P(\parallel) \propto u_z^2$ and $P(\perp) \propto u_y^2$, while the $u_x^2$ component is not collected (it radiates along the detection axis). This keeps the count-based anisotropy of the two streams exactly $r(t) = 0.4\,e^{-t/\tau_{rot}}$.


In [ ]:
def sample_excited_orientations(n, rs):
    """Orientation of the excited transition dipole after absorption of a
    z-polarized photon: p(cos theta) ~ cos^2 theta (=> r0 = 0.4)"""
    x = rs.rand(n)
    cos_t = np.cbrt(2.0 * x - 1.0)
    sin_t = np.sqrt(np.maximum(0.0, 1.0 - cos_t**2))
    phi = 2.0 * np.pi * rs.rand(n)
    u = np.empty((n, 3))
    u[:, 0] = sin_t * np.cos(phi)
    u[:, 1] = sin_t * np.sin(phi)
    u[:, 2] = cos_t
    return u

def random_axes(n, rs):
    v = rs.normal(size=(n, 3))
    return v / np.linalg.norm(v, axis=1)[:, None]

def rotate(u, axes, angles):
    """Rodrigues rotation of u around unit axes by angles (radians)"""
    c = np.cos(angles)[:, None]
    s = np.sin(angles)[:, None]
    dot = np.sum(u * axes, axis=1)[:, None]
    return u * c + np.cross(axes, u) * s + axes * dot * (1.0 - c)

def rot_diffuse(u, D_rot, t, rs):
    """
    Exact single-time propagator of isotropic rotational diffusion over the delay
    time t (ns), with D_rot in 1/ns. The P2 autocorrelation decays as
    exp(-6 D_rot t) = exp(-t / tau_rot): with that probability the dipole has
    (not yet) diffusively decorrelated from its initial orientation, otherwise
    the orientation is fully thermalized, i.e. isotropic.
    """
    t = np.asarray(t, dtype = np.float64)
    keep = rs.rand(u.shape[0]) < np.exp(-6.0 * D_rot * t)
    out = u.copy()
    out[~keep] = random_axes((~keep).sum(), rs)
    return out

def sort_polarization(u, rs):
    """
    Sort photons into the parallel (z) and perpendicular (y) polarization
    channels of a detector viewing along x. A dipole emits no light along
    its own axis, so the emission towards the detector is polarized in the
    plane perpendicular to x: I_par ~ u_z^2, I_perp ~ u_y^2, and the u_x^2
    component radiates along the detection axis and is NOT collected.
    Returns the channel of each photon: 0 = parallel, 1 = perpendicular,
    -1 = not collected by the polarization-resolved detector.
    """
    draw = rs.rand(u.shape[0])
    I_par = u[:, 2]**2
    I_perp = u[:, 1]**2
    channel = np.full(u.shape[0], -1, dtype = np.int64)
    channel[draw < I_par + I_perp] = 0
    channel[(draw >= I_par) & (draw < I_par + I_perp)] = 1
    return channel

def r_from_streams(I_par, I_perp):
    """Ensemble anisotropy from the two polarization intensities"""
    return (I_par - I_perp) / (I_par + 2.0 * I_perp)


In [ ]:
# Quick validation of the physical model:
# r(0) should be 0.4 and r(t) should follow the Perrin equation
rs_check = np.random.RandomState(42)
N_check = 200_000
tau_rot_check = 1.0 # ns
D_rot_check = 1.0/(6.0*tau_rot_check)

u0 = sample_excited_orientations(N_check, rs_check)
ch0 = sort_polarization(u0, rs_check)
n_par = (ch0 == 0).sum(); n_perp = (ch0 == 1).sum()
print(f"r(0) = {r_from_streams(n_par, n_perp):.3f} (expected 0.400)")

t_check = np.linspace(0, 3, 7)
r_check = []
for t in t_check:
    u_t = rot_diffuse(u0, D_rot_check, t, rs_check)
    ch_t = sort_polarization(u_t, rs_check)
    n_par = (ch_t == 0).sum(); n_perp = (ch_t == 1).sum()
    r_check.append(r_from_streams(n_par, n_perp))
r_check = np.array(r_check)

plt.plot(t_check, r_check, 'o', label = "simulation")
plt.plot(t_check, 0.4*np.exp(-t_check/tau_rot_check), '-', label = r"$r_0 e^{-t/	au_{rot}}$")
plt.xlabel("time / ns")
plt.ylabel("r(t)")
plt.legend(); plt.title("Perrin decay of the simulated anisotropy"); plt.show()


## 4. Dye parameters

Each dye (donor and acceptor) gets its own fluorescence lifetime, rotational correlation time and the relative orientation of its absorption and emission dipoles. The donor of a FRET pair is excited (Dex), transfers to the acceptor, which then emits. The simplest model assumes the acceptor emission dipole is rigidly attached to the donor absorption dipole (fixed angle `donor_acceptor_angle`, i.e. a fixed $\kappa^2$-like coupling geometry), and both dyes wobble independently while in their respective excited states.


In [ ]:
# Anisotropy model parameters
base_lifetime = 3.0 # ns, as in notebook 3
E = E_fret

tau_D = -base_lifetime*(E - 1.0)          # Donor lifetime from FRET efficiency, as in notebook 3
tau_A = base_lifetime                     # Acceptor lifetime (under direct acceptor excitation)
tau_rot_D = 1.5                           # Donor rotational correlation time / ns (flexible linker)
tau_rot_A = 1.5                           # Acceptor rotational correlation time / ns
D_rot_D = 1.0/(6.0*tau_rot_D)
D_rot_A = 1.0/(6.0*tau_rot_A)
donor_acceptor_angle = np.deg2rad(60.0)    # Angle between donor absorption and acceptor emission dipole

print(f"Donor lifetime: {tau_D:.2f} ns, acceptor lifetime: {tau_A:.2f} ns")
print(f"tau_rot donor: {tau_rot_D} ns, tau_rot acceptor: {tau_rot_A} ns")


## 5. Load the PIE photon streams and sort them by polarization

The emission simulations of section 1 provide the timestamps, detector IDs and particle IDs. As in notebook 2, the DexDem/DexAem simulation is combined with the acceptor-only simulation. As in notebook 3, DexAem and AexAem are distinguished by temporary nanotimes (5 for DexAem, 2 for AexAem) before the real nanotimes are simulated here.

Then every photon is assigned a physical polarization:

- **DexDem (donor emission)**: the donor absorbs ($\cos^2\theta$-distributed orientation), wobbles for the donor delay time, emits into $\parallel$/$\perp$.
- **DexAem (FRET, acceptor emission after donor excitation)**: the donor absorbs, wobbles for the donor lifetime (that is when FRET happens), then the acceptor emission dipole (rigidly attached to the donor dipole at `donor_acceptor_angle`) wobbles for the remaining acceptor delay time and emits into $\parallel$/$\perp$.
- **AexAem (direct acceptor excitation)**: the acceptor absorbs, wobbles for the acceptor delay time, emits into $\parallel$/$\perp$.


In [ ]:
def rotate_around_z(u, angle):
    """Rigid rotation around the z axis by a fixed angle"""
    c, s = np.cos(angle), np.sin(angle)
    u_rot = u.copy()
    u_rot[:, 0] = c*u[:, 0] - s*u[:, 1]
    u_rot[:, 1] = s*u[:, 0] + c*u[:, 1]
    return u_rot

def rotate_around_y(u, angle):
    """Rigid rotation around the y axis by a fixed angle"""
    c, s = np.cos(angle), np.sin(angle)
    u_rot = u.copy()
    u_rot[:, 0] = c*u[:, 0] + s*u[:, 2]
    u_rot[:, 2] = -s*u[:, 0] + c*u[:, 2]
    return u_rot


In [ ]:
# Stack the DexDem/DexAem and the AexAem simulation, as in make_pie_hdf5() of notebook 2
fret_file = [f for f in fret_sim.m_filenames if f"_{E_fret:.0%}"[:-1] + f"{int(E_fret*100):02d}" in f or f"E_{int(E_fret*100):02d}" in f][0]
acceptor_file = a_only_sim.m_filenames[0]

fret_array = phc.hdf5.load_photon_hdf5(fret_file)
acceptor_array = phc.hdf5.load_photon_hdf5(acceptor_file)

new_timestamps = np.hstack([
    fret_array.photon_data.timestamps.read(),
    acceptor_array.photon_data.timestamps.read()
])
new_detectors = np.hstack([
    fret_array.photon_data.detectors.read(),
    acceptor_array.photon_data.detectors.read()
])
new_particles = np.hstack([
    fret_array.photon_data.particles.read(),
    acceptor_array.photon_data.particles.read()
])

# Temporary nanotimes to tell DexAem (5) from AexAem (2), as in notebook 2
new_nanotimes = np.hstack(
    [5*np.ones(fret_array.photon_data.detectors.read().shape[0], dtype = np.uint16),
     2*np.ones(acceptor_array.photon_data.detectors.read().shape[0], dtype = np.uint16)],
    dtype = np.uint16
)

argsort = new_timestamps.argsort(kind='mergeset' if False else 'mergesort')
new_timestamps = new_timestamps[argsort]
new_detectors = new_detectors[argsort]
new_particles = new_particles[argsort]
new_nanotimes = new_nanotimes[argsort]

times_unit = fret_array.photon_data.timestamps_specs.timestamps_unit.read()
print(f"{new_timestamps.size} photons, times unit: {times_unit} s")


In [ ]:
def simulate_anisotropy(timestamps, det, par, temp_nanotimes, rs,
                        tau_D, tau_A, D_rot_D, D_rot_A, donor_acceptor_angle,
                        background_par = False):
    """
    Assign physically simulated polarization channels and nanotimes to every photon.
    ---
    timestamps, det, par, temp_nanotimes: photon arrays (det 0 = Dex streams, det 1 = AexAem)
    temp_nanotimes: 5 = DexAem, 2 = AexAem (convention of notebooks 2 and 3)
    Returns: new detectors (0 = D parallel, 1 = D perpendicular, 2 = A parallel,
             3 = A perpendicular, 255 = not collected by the polarization detector),
             new nanotimes, and the per-photon delay times used for the rotation
    """
    n = timestamps.size
    det_new = np.zeros(n, dtype = np.uint8)
    nano_new = np.zeros(n, dtype = np.uint16)
    delay_ns = np.zeros(n)

    # --- Donor emission photons (DexDem): detector 0 ---
    mask_dem = (det == 0)
    n_dem = mask_dem.sum()
    nano_dem, delay_dem = rand_nanotimes(tau_D, n_dem, rs)
    delay_ns[mask_dem] = delay_dem
    u = sample_excited_orientations(n_dem, rs)
    u = rot_diffuse(u, D_rot_D, delay_ns[mask_dem], rs)
    channel = sort_polarization(u, rs)
    det_new[mask_dem] = np.where(channel == 0, 0, 1)
    det_new[np.where(mask_dem)[0][channel < 0]] = 255 # not collected
    nano_new[mask_dem] = nano_dem

    # --- FRET photons (DexAem): detector 1, temporary nanotime 5 ---
    mask_dexaem = (det == 1) & (temp_nanotimes == 5)
    n_fret = mask_dexaem.sum()
    # Donor excited state: absorption + wobbling until energy transfer
    lt_D = rs.exponential(scale = tau_D, size = n_fret) # ns, time until FRET
    u_D = sample_excited_orientations(n_fret, rs)
    u_D = rot_diffuse(u_D, D_rot_D, lt_D, rs)
    # Acceptor emission dipole: rigidly attached to the donor dipole
    u_A = rotate_around_y(u_D, donor_acceptor_angle)
    # Acceptor delay: remaining excited state lifetime
    lt_A = rs.exponential(scale = tau_A, size = n_fret) # ns
    u_A = rot_diffuse(u_A, D_rot_A, lt_A, rs)
    channel = sort_polarization(u_A, rs)
    det_new[mask_dexaem] = np.where(channel == 0, 2, 3)
    det_new[np.where(mask_dexaem)[0][channel < 0]] = 255 # not collected
    # Total nanotime: donor decay until FRET + acceptor decay, IRF-convolved
    delay_fret = (lt_D + lt_A)
    delay_ns[mask_dexaem] = delay_fret
    nano_new[mask_dexaem] = ((IRF_CENTER + np.rint(delay_fret/TCSPC_UNIT_NS).astype(np.int64))
                             % NUM_BINS).astype(np.uint16)

    # --- Direct acceptor excitation (AexAem): detector 1, temporary nanotime 2 ---
    mask_aexaem = (det == 1) & (temp_nanotimes == 2)
    n_aex = mask_aexaem.sum()
    nano_aex, delay_aex = rand_nanotimes(tau_A, n_aex, rs)
    delay_ns[mask_aexaem] = delay_aex
    u = sample_excited_orientations(n_aex, rs)
    u = rot_diffuse(u, D_rot_A, delay_ns[mask_aexaem], rs)
    channel = sort_polarization(u, rs)
    det_new[mask_aexaem] = np.where(channel == 0, 2, 3)
    det_new[np.where(mask_aexaem)[0][channel < 0]] = 255 # not collected
    nano_new[mask_aexaem] = nano_aex

    # --- Background photons: unpolarized, uniform nanotimes ---
    mask_bg = (par == par.max()) # PyBroMo's last "particle" is the background
    det_new[mask_bg] = rs.choice([0, 1, 2, 3], mask_bg.sum(), p = [0.25, 0.25, 0.25, 0.25])
    nano_new[mask_bg] = rs.randint(0, 4096, mask_bg.sum()).astype(np.uint16)
    delay_ns[mask_bg] = -1 # flag, no anisotropy

    return det_new, nano_new, delay_ns


In [ ]:
rs_aniso = np.random.RandomState(seed = 73199)
det_aniso, nano_aniso, delay_ns = simulate_anisotropy(
    new_timestamps, new_detectors, new_particles, new_nanotimes, rs_aniso,
    tau_D, tau_A, D_rot_D, D_rot_A, donor_acceptor_angle
)

streams, counts = np.unique(det_aniso, return_counts = True)
names = ["D parallel", "D perpendicular", "A parallel", "A perpendicular", "not collected"]
for s, c in zip(streams, counts):
    print(f"{names[s]}: {c} photons")

# Photons whose emission dipole points along the detection axis (detector id 255)
# are not collected: a dipole radiates no light along its own axis. They are
# dropped from the final photon streams.
collected = det_aniso != 255
print("Collected fraction:", round(collected.mean(), 2), "(2/3 expected for isotropic dyes)")


## 6. Save as polarization-resolved photon-HDF5

The four streams are written to a photon-HDF5 file. Compared to notebooks 2 and 3, the setup now declares `num_polarization_ch = 2`, and the detectors map onto the four combinations of spectral channel (donor/acceptor) and polarization channel (parallel/perpendicular).


In [ ]:
def make_anisotropy_photon_hdf5(times, det, par, nanotimes, times_unit, description, identity=None):
    photon_data = dict(
        timestamps = times,
        timestamps_specs = dict(timestamps_unit=times_unit),
        detectors = det,
        particles = par,
        nanotimes = nanotimes,
        nanotimes_specs = {
            'tcspc_unit': (1/LASER_RATE)/4096,
            'tcspc_range': (1/LASER_RATE),
            'tcspc_num_bins': 4096
        },
        measurement_specs = dict(
            measurement_type = 'smFRET-nsALEX',
            laser_repetition_rate = LASER_RATE,
            detectors_specs = dict(spectral_ch1 = np.atleast_1d(0),
                                   spectral_ch2 = np.atleast_1d(1),
                                   polarization_ch1 = np.atleast_1d(0),
                                   polarization_ch2 = np.atleast_1d(1))))

    setup = dict(
        num_pixels = 1,
        num_spots = 1,
        num_spectral_ch = 2,
        num_polarization_ch = 2,
        num_split_ch = 1,
        modulated_excitation = True,
        lifetime = True,
        laser_repetition_rates = [LASER_RATE],
        excitation_alternated=(True,),
        excitation_cw=(False,)
    )

    provenance = dict(software='', software_version='', filename='')

    if identity is None:
        identity = dict()

    acquisition_duration = ((times[-1] - times[0]) * times_unit)
    data = dict(
        acquisition_duration = round(acquisition_duration),
        description = description,
        photon_data = photon_data,
        setup=setup,
        provenance=provenance,
        identity=identity)
    return data

description = "Simulated polarization-resolved (anisotropy) PIE-smFRET data"
identity = dict(author='Max Mustermann', author_affiliation='Simulated Data')
times_out = new_timestamps[collected]
det_out = det_aniso[collected].astype(np.uint8)
par_out = new_particles[collected]
nano_out = nano_aniso[collected]
order = np.argsort(times_out, kind = 'mergesort')
data = make_anisotropy_photon_hdf5(
    times_out[order], det_out[order], par_out[order], nano_out[order], times_unit,
    description, identity=identity
)

h5_fname = (f"smFRET_{sim_id}_E{int(E_fret*100)}_aniso_tauD{tau_rot_D}ns_"
           f"tauA{tau_rot_A}ns_pie.h5")
print(h5_fname)
phc.hdf5.save_photon_hdf5(data, h5_fname = h5_fname, overwrite = True)


## 7. Verify: time-resolved anisotropy decay

From the nanotime histograms of the parallel and perpendicular streams of each dye, the time-resolved anisotropy

$$r(t) = \frac{I_{\parallel}(t) - I_{\perp}(t)}{I_{\parallel}(t) + 2 I_{\perp}(t)}$$

is computed and compared to the Perrin equation $r(t) = r_0 e^{-t/\tau_{rot}}$. The perpendicular decays are also $G$-factor corrected in the usual way; for this simulation $G = 1$ by construction.


In [ ]:
def anisotropy_decay(nano_par, nano_perp, n_bins = 64, g_factor = 1.0, tail_mask = None):
    """Time-resolved anisotropy from nanotime histograms of the two polarization streams"""
    hist_par, edges = np.histogram(nano_par, bins = n_bins, range = (0, 4096))
    hist_perp, _ = np.histogram(nano_perp, bins = n_bins, range = (0, 4096))
    # Time axis relative to the IRF peak, so t = 0 is the excitation pulse
    t = (0.5*(edges[:-1] + edges[1:]) - IRF_CENTER) * TCSPC_UNIT_NS # ns
    with np.errstate(invalid='ignore', divide='ignore'):
        r = (hist_par - g_factor*hist_perp) / (hist_par + 2*g_factor*hist_perp)
    valid = (hist_par + hist_perp) > 50 # only bins with enough photons
    return t[valid], r[valid]

# Donor streams
t_D, r_D = anisotropy_decay(
    nano_aniso[(det_aniso == 0) & (delay_ns >= 0)],
    nano_aniso[(det_aniso == 1) & (delay_ns >= 0)]
)
# Acceptor streams (DexAem + AexAem combined)
t_A, r_A = anisotropy_decay(
    nano_aniso[(det_aniso == 2) & (delay_ns >= 0)],
    nano_aniso[(det_aniso == 3) & (delay_ns >= 0)]
)

fig, ax = plt.subplots(1, 2, figsize = (10, 4), sharey = True)
ax[0].plot(t_D, r_D, 'o', label = "simulated donor r(t)")
ax[0].plot(t_D, 0.4*np.exp(-t_D/tau_rot_D), '-', label = r"$0.4\,e^{-t/	au_{rot,D}}$")
ax[0].set_xlabel("time / ns"); ax[0].set_ylabel("r(t)"); ax[0].legend()
ax[0].set_title(f"Donor, $\tau_{{rot}}$ = {tau_rot_D} ns")
ax[1].plot(t_A, r_A, 'o', label = "simulated acceptor r(t)")
ax[1].plot(t_A, 0.4*np.exp(-t_A/tau_rot_A), '-', label = r"$0.4\,e^{-t/	au_{rot,A}}$")
ax[1].set_xlabel("time / ns"); ax[1].legend()
ax[1].set_title(f"Acceptor, $\tau_{{rot}}$ = {tau_rot_A} ns")
plt.tight_layout(); plt.show()


In [ ]:
# The fluorescence decay in the two polarization channels, and the steady state anisotropy
fig, ax = plt.subplots(1, 2, figsize = (10, 4))
for stream, name, color in [(0, "D $\parallel$", "C0"), (1, "D $\perp$", "C1")]:
    ax[0].hist(nano_aniso[(det_aniso == stream) & (delay_ns >= 0)], bins = 128,
               histtype = u"step", lw = 2, color = color, label = name, density = True)
ax[0].set_xlabel("nanotime / bins"); ax[0].set_ylabel("normalized counts"); ax[0].legend()
ax[0].set_title("Donor polarized decays")
for stream, name, color in [(2, "A $\parallel$", "C0"), (3, "A $\perp$", "C1")]:
    ax[1].hist(nano_aniso[(det_aniso == stream) & (delay_ns >= 0)], bins = 128,
               histtype = u"step", lw = 2, color = color, label = name, density = True)
ax[1].set_xlabel("nanotime / bins"); ax[1].legend()
ax[1].set_title("Acceptor polarized decays")
plt.tight_layout(); plt.show()

# Steady-state anisotropy (corrected for background-free photons only)
for label, d_par, d_perp in [
    ("Donor", 0, 1), ("Acceptor", 2, 3)
]:
    mask_par = (det_aniso == d_par) & (delay_ns >= 0)
    mask_perp = (det_aniso == d_perp) & (delay_ns >= 0)
    I_par, I_perp = mask_par.sum(), mask_perp.sum()
    print(f"{label} steady-state anisotropy: {r_from_streams(I_par, I_perp):.3f}")


## 8. Burst-wise anisotropy, as it would be measured

The simulated file can be loaded in FRETBursts like the files of notebook 2. The four streams appear as four detectors, so burst-wise anisotropy $r = (I_\parallel - I_\perp)/(I_\parallel + 2I_\perp)$ can be computed per burst and per species, which is the input needed for H2MM-type multi-state analysis of polarization-resolved data.


In [ ]:
# Load the polarization-resolved simulation with fretbursts (the D_ON/A_ON windows
# need to be adapted to the four-detector scheme of this simulation)
d = fb.loader.photon_hdf5(h5_fname)
print("Detectors in file:", np.unique(d.ph_data_m.get('detectors', det_aniso)))

# Burst search on the summed signal
d.add(D_ON=(100, 2100), A_ON=(2150, 4095))
fb.loader.alex_apply_period(d)
d.calc_bg(fun = fb.bg.exp_fit, time_s = 30, F_bg = 1.7)
d.burst_search(F = 6, m = 5)
d.fuse_bursts(ms = 0)
print(f"Number of bursts: {d.num_bursts}")

# Burst-wise anisotropy of the donor streams
nd_par = d.nd + d.na # placeholder: adapt the detector routing to your setup
print("Adapt the D_ON/A_ON and detector routing to extract the polarization streams per burst.")


## Summary

- Starting from the PIE emission simulations of notebook 2 (single-particle emission through the PSF, donor and acceptor streams, PIE combination), every photon was reassigned to a **parallel or perpendicular polarization stream** based on a physical single-molecule anisotropy simulation.
- The dye reorientation model is the simplest possible: $\cos^2\theta$ photoselection by the polarized laser pulse, isotropic Brownian rotational diffusion ($\tau_{rot} = 1/(6 D_{rot})$) during the excited-state lifetime, and dipole emission probabilities into two analyzers.
- The time-resolved machinery (IRF convolution, exponential lifetime draws, 4096-bin nanotimes at 20 MHz) is taken directly from notebook 3, so the simulated anisotropy decays are convolved with the instrument response just like real TCSPC data.
- Validation: $r(0) = 0.4$, Perrin decay $r(t) = 0.4 e^{-t/\tau_{rot}}$ for both dyes, and the output is a photon-HDF5 file declaring two polarization channels with four detector streams.
